# 01. ByT5 Exact-Match Baseline

이 노트북은 Track A, 즉 official ERR-compatible ByT5 baseline을 실행합니다.

- `ufal/byt5-small-multilexnorm2021-en` checkpoint에서 시작합니다.
- 새 MultiLexNorm++ 언어 `id, ja, ko, th, vi`를 언어별로 따로 fine-tuning합니다.
- 입력은 official target-token 형식입니다: `sentence <extra_id_0> target <extra_id_1>`.
- 출력은 target token 하나의 normalized form입니다.
- 기본 설정은 전체 5개 언어를 3 epoch씩 먼저 학습하는 축소 실행입니다.

주요 출력:

- `outputs/target_examples/{lang}`
- `outputs/byt5/ByT52021EN_to_{lang}/predictions.csv`
- `outputs/byt5/ByT52021EN_to_{lang}/summary.csv`

Drive에는 같은 구조로 `/drive/MyDrive/AI개론_박진영/lexnorm_outputs/...`가 mirror 저장됩니다.

나중에 시간이 되면 별도 셀에서 저장된 3ep 모델을 불러와 7 epoch를 추가 학습할 수 있습니다. ByT5는 안정성을 위해 `--fp16 false`를 유지합니다. `00`에서 `WANDB_API_KEY`가 로드되어 있으면 fine-tuning loss/lr/eval_loss가 W&B에 기록됩니다.


## 실험 설정

데이터셋 이름, fine-tuning 대상 언어, target-token dataset 출력 위치를 정의합니다.

이 노트북은 shell command가 아니라 Python function call로 실행합니다. 실패 시 `CalledProcessError`가 아니라 실제 함수 내부 traceback이 보이도록 하기 위함입니다. `TRAIN_EXAMPLES_PER_EPOCH = 50_000`으로 고정하여 언어별 train 크기가 달라도 1 epoch의 학습량을 맞춥니다. 현재 기본 batch는 `train_batch=25`, `grad_accum=4`, effective batch `100`입니다. `EVAL_BATCH_SIZE`와 `PRED_BATCH_SIZE`는 별도로 조절합니다.


In [4]:
import pathlib, runpy

BOOTSTRAP = pathlib.Path("scripts/01_02_03_04_05_06_notebook_bootstrap.py")
if not BOOTSTRAP.exists():
    BOOTSTRAP = pathlib.Path("/content/lexnorm_submit/scripts/01_02_03_04_05_06_notebook_bootstrap.py")

setup_project = runpy.run_path(str(BOOTSTRAP))["setup_project"]
PROJECT_ROOT = setup_project()

from lexnorm.data import build_target_examples_per_lang
from lexnorm.seq2seq import evaluate_prediction_csv, fine_tune_seq2seq, predict_seq2seq
from lexnorm.utils import drive_output_root, sync_to_drive

DATASET = "weerayut/multilexnorm2026-dev-pub"
LANGS = ["id", "ja", "ko", "th", "vi"]
OUT_ROOT = "outputs/target_examples"
WANDB_PROJECT = "AI_Intro_lexnorm"
WANDB_ENTITY = "wjdrldyd0213-sungkyunkwan-university"
LOGGING_STEPS = 10
TRAIN_EXAMPLES_PER_EPOCH = 50_000
TRAIN_BATCH_SIZE = 20
EVAL_BATCH_SIZE = 16
PRED_BATCH_SIZE = 16
GRAD_ACCUM = 5
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM


def estimate_updates(lang, epochs):
    from datasets import load_from_disk
    raw_train_len = len(load_from_disk(f"{OUT_ROOT}/{lang}")["train"])
    sampled_train_len = TRAIN_EXAMPLES_PER_EPOCH or raw_train_len
    updates_per_epoch = -(-sampled_train_len // EFFECTIVE_BATCH_SIZE)
    return raw_train_len, sampled_train_len, updates_per_epoch, int(updates_per_epoch * epochs)

print(f"train_batch={TRAIN_BATCH_SIZE}, grad_accum={GRAD_ACCUM}, effective_batch={EFFECTIVE_BATCH_SIZE}")
print(f"train_examples_per_epoch={TRAIN_EXAMPLES_PER_EPOCH:,}")

print("Drive mirror root =", drive_output_root())


PROJECT_ROOT = /content/lexnorm_submit
train_batch=20, grad_accum=5, effective_batch=100
train_examples_per_epoch=50,000
Drive mirror root = /drive/MyDrive/AI개론_박진영/lexnorm_outputs


## target-token dataset 생성

각 언어별로 `<extra_id_0>`/`<extra_id_1>`가 들어간 학습 예제를 만들고 `outputs/target_examples/{lang}`에 저장합니다. 여기서는 `scripts/`를 subprocess로 실행하지 않고 `lexnorm.data.build_target_examples_per_lang()`을 직접 호출합니다.


In [5]:
summary = build_target_examples_per_lang(
    dataset_name=DATASET,
    output_root=OUT_ROOT,
    langs=LANGS,
    add_lang_prefix=False,
    target_filter="alnum",
)
sync_to_drive(OUT_ROOT)
summary


actual dataset languages: ['da', 'de', 'en', 'es', 'hr', 'id', 'iden', 'it', 'ja', 'ko', 'nl', 'sl', 'sr', 'th', 'tr', 'trde', 'vi']


Saving the dataset (0/1 shards):   0%|          | 0/28372 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3411 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3675 [00:00<?, ? examples/s]

id {'train': 28372, 'validation': 3411, 'test': 3675} -> outputs/target_examples/id


...

Saving the dataset (0/1 shards):   0%|          | 0/96536 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/12924 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/6004 [00:00<?, ? examples/s]

vi {'train': 96536, 'validation': 12924, 'test': 6004} -> outputs/target_examples/vi
[drive sync] outputs/target_examples -> /drive/MyDrive/AI개론_박진영/lexnorm_outputs/target_examples


{'id': {'train': 28372, 'validation': 3411, 'test': 3675},
 'ja': {'train': 54065, 'validation': 9596, 'test': 9287},
 'ko': {'train': 13109, 'validation': 1878, 'test': 822},
 'th': {'train': 41511, 'validation': 5912, 'test': 6038},
 'vi': {'train': 96536, 'validation': 12924, 'test': 6004}}

## VRAM 사전 점검

현재 batch/accum/eval/predict 설정으로 한 batch dry-run을 실행해 학습/eval/predict peak VRAM을 측정합니다. 전체 학습 전에 OOM 가능성을 확인하는 용도입니다. 측정은 선택한 한 언어의 샘플 batch 기준이므로, 더 긴 문장이 섞인 batch에서는 peak가 조금 더 올라갈 수 있습니다.


In [6]:
RUN_VRAM_PROBE = True
VRAM_PROBE_LANG = LANGS[0]


def _gb(x):
    return x / 1024**3


if RUN_VRAM_PROBE:
    import gc
    import torch
    from datasets import load_from_disk
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    from transformers.optimization import Adafactor

    if not torch.cuda.is_available():
        print("CUDA is not available; VRAM probe skipped.")
    else:
        device = "cuda"
        ds_probe = load_from_disk(f"{OUT_ROOT}/{VRAM_PROBE_LANG}")
        tokenizer = AutoTokenizer.from_pretrained("ufal/byt5-small-multilexnorm2021-en")
        model = AutoModelForSeq2SeqLM.from_pretrained("ufal/byt5-small-multilexnorm2021-en").to(device)
        optimizer = Adafactor(model.parameters(), lr=1e-4, scale_parameter=False, relative_step=False)

        def make_batch(split, batch_size):
            split_ds = ds_probe[split]
            batch_size = min(batch_size, len(split_ds))
            rows = [split_ds[i] for i in range(batch_size)]
            enc = tokenizer(
                [row["input_text"] for row in rows],
                padding=True,
                truncation=True,
                max_length=200,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            labels = tokenizer(
                text_target=[row["target_text"] for row in rows],
                padding=True,
                truncation=True,
                max_length=32,
                return_tensors="pt",
            )["input_ids"].to(device)
            labels[labels == tokenizer.pad_token_id] = -100
            enc["labels"] = labels
            return enc

        def measure_train():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            model.train()
            enc = make_batch("train", TRAIN_BATCH_SIZE)
            loss = model(**enc).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            return _gb(torch.cuda.max_memory_allocated()), _gb(torch.cuda.max_memory_reserved())

        def measure_generate(split, batch_size):
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            model.eval()
            enc = make_batch(split, batch_size)
            labels = enc.pop("labels")
            del labels
            with torch.no_grad():
                _ = model.generate(**enc, num_beams=1, max_new_tokens=32, do_sample=False)
            return _gb(torch.cuda.max_memory_allocated()), _gb(torch.cuda.max_memory_reserved())

        print(f"VRAM probe language={VRAM_PROBE_LANG}")
        print(f"train batch={TRAIN_BATCH_SIZE}, grad_accum={GRAD_ACCUM}, effective={EFFECTIVE_BATCH_SIZE}")
        print(f"eval batch={EVAL_BATCH_SIZE}, predict batch={PRED_BATCH_SIZE}")
        print(f"GPU total={_gb(torch.cuda.get_device_properties(0).total_memory):.2f} GB")

        try:
            train_alloc, train_reserved = measure_train()
            print(f"train one-step peak: allocated={train_alloc:.2f} GB, reserved={train_reserved:.2f} GB")
            eval_alloc, eval_reserved = measure_generate("validation", EVAL_BATCH_SIZE)
            print(f"eval generation peak: allocated={eval_alloc:.2f} GB, reserved={eval_reserved:.2f} GB")
            pred_alloc, pred_reserved = measure_generate("validation", PRED_BATCH_SIZE)
            print(f"predict generation peak: allocated={pred_alloc:.2f} GB, reserved={pred_reserved:.2f} GB")
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                print("CUDA OOM during probe. Lower TRAIN_BATCH_SIZE or EVAL_BATCH_SIZE before full training.")
                print(exc)
            else:
                raise
        finally:
            del model, tokenizer, optimizer
            gc.collect()
            torch.cuda.empty_cache()
else:
    print("VRAM probe disabled. Set RUN_VRAM_PROBE=True to measure before training.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


VRAM probe language=id
train batch=20, grad_accum=5, effective=100
eval batch=16, predict batch=16
GPU total=14.56 GB
train one-step peak: allocated=3.47 GB, reserved=4.47 GB
eval generation peak: allocated=1.29 GB, reserved=1.45 GB
predict generation peak: allocated=1.29 GB, reserved=1.45 GB


## ByT5 3 epoch fine-tuning 실행

5개 언어를 각각 3 epoch씩 학습합니다. sampler 때문에 각 언어의 1 epoch는 50,000 examples로 계산됩니다. `lexnorm.seq2seq.fine_tune_seq2seq()`를 직접 호출합니다.


In [7]:
LANGS_TO_RUN = ['vi']
EPOCHS = 3

print(f"Fine-tuning tasks: {len(LANGS_TO_RUN)} language(s) -> {LANGS_TO_RUN}")
for task_idx, lang in enumerate(LANGS_TO_RUN, start=1):
    raw_train_len, sampled_train_len, updates_per_epoch, total_updates = estimate_updates(lang, EPOCHS)
    print("=" * 80)
    print(f"[{task_idx}/{len(LANGS_TO_RUN)}] training ByT52021EN_to_{lang}")
    print(
        f"train pool={raw_train_len:,}, sampled/epoch={sampled_train_len:,}, "
        f"epochs={EPOCHS}, updates/epoch~{updates_per_epoch:,}, total updates~{total_updates:,}"
    )

    fine_tune_seq2seq(
        model_name_or_path="ufal/byt5-small-multilexnorm2021-en",
        data_dir=f"{OUT_ROOT}/{lang}",
        output_dir=f"outputs/byt5/ByT52021EN_to_{lang}",
        num_train_epochs=EPOCHS,
        learning_rate=1e-4,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        train_examples_per_epoch=TRAIN_EXAMPLES_PER_EPOCH,
        max_source_length=200,
        max_target_length=32,
        fp16=False,
        lr_scheduler_type="constant",
        wandb_project=WANDB_PROJECT,
        wandb_entity=WANDB_ENTITY,
        wandb_group="ByT52021EN_transfer_3ep",
        wandb_run_name=f"ByT52021EN_to_{lang}_3ep",
        logging_steps=LOGGING_STEPS,
    )
    sync_to_drive(f"outputs/byt5/ByT52021EN_to_{lang}")
    print(f"[{task_idx}/{len(LANGS_TO_RUN)}] done: ByT52021EN_to_{lang} 3ep")


Fine-tuning tasks: 1 language(s) -> ['vi']
[1/1] training ByT52021EN_to_vi
train pool=96,536, sampled/epoch=50,000, epochs=3, updates/epoch~500, total updates~1,500
[train] model=ufal/byt5-small-multilexnorm2021-en data=outputs/target_examples/vi output=outputs/byt5/ByT52021EN_to_vi train_pool=96536 train_examples_per_epoch=50000 eval=12924 epochs=3 batch=20 grad_accum=5 effective_batch=100 updates_per_epoch~500 total_updates~1500
[train] loading tokenizer/model


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[train] tokenizing train/eval splits
[train] tokenization complete
[train] trainer.train() start


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: wjdrldyd0213 (wjdrldyd0213-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,4.192406,0.610768
2,1.616406,0.238749
3,0.764661,0.152030


[train] trainer.train() complete


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[train] saved final model -> outputs/byt5/ByT52021EN_to_vi
[drive sync] outputs/byt5/ByT52021EN_to_vi -> /drive/MyDrive/AI개론_박진영/lexnorm_outputs/byt5/ByT52021EN_to_vi
[1/1] done: ByT52021EN_to_vi 3ep


## 선택: 나머지 7 epoch continued fine-tuning

3ep 학습이 끝난 뒤 시간이 남으면 이 셀에서 `RUN_EXTRA_7EP = True`로 바꾸고 실행합니다. 같은 모델 폴더에서 이어서 7 epoch를 더 학습합니다. 여기서도 1 epoch는 sampler 기준 50,000 examples입니다.


In [8]:
# RUN_EXTRA_7EP = False
# EXTRA_EPOCHS = 7
# EXTRA_LANGS_TO_RUN = LANGS

# if not RUN_EXTRA_7EP:
#     print("Extra 7ep is disabled. Set RUN_EXTRA_7EP = True when you want to continue from saved 3ep models.")
# else:
#     print(f"Extra fine-tuning tasks: {len(EXTRA_LANGS_TO_RUN)} language(s) -> {EXTRA_LANGS_TO_RUN}")
#     for task_idx, lang in enumerate(EXTRA_LANGS_TO_RUN, start=1):
#         model_dir = pathlib.Path(f"outputs/byt5/ByT52021EN_to_{lang}")
#         if not (model_dir / "train_complete.json").exists():
#             raise FileNotFoundError(f"3ep model is missing or incomplete: {model_dir}")

#         raw_train_len, sampled_train_len, updates_per_epoch, total_updates = estimate_updates(lang, EXTRA_EPOCHS)
#         print("=" * 80)
#         print(f"[{task_idx}/{len(EXTRA_LANGS_TO_RUN)}] extra training ByT52021EN_to_{lang}")
#         print(
#             f"train pool={raw_train_len:,}, sampled/epoch={sampled_train_len:,}, "
#             f"extra_epochs={EXTRA_EPOCHS}, updates/epoch~{updates_per_epoch:,}, extra updates~{total_updates:,}"
#         )

#         fine_tune_seq2seq(
#             model_name_or_path=str(model_dir),
#             data_dir=f"{OUT_ROOT}/{lang}",
#             output_dir=str(model_dir),
#             num_train_epochs=EXTRA_EPOCHS,
#             learning_rate=1e-4,
#             per_device_train_batch_size=TRAIN_BATCH_SIZE,
#             per_device_eval_batch_size=EVAL_BATCH_SIZE,
#             gradient_accumulation_steps=GRAD_ACCUM,
#             train_examples_per_epoch=TRAIN_EXAMPLES_PER_EPOCH,
#             max_source_length=200,
#             max_target_length=32,
#             fp16=False,
#             lr_scheduler_type="constant",
#             wandb_project=WANDB_PROJECT,
#             wandb_entity=WANDB_ENTITY,
#             wandb_group="ByT52021EN_transfer_extra7ep",
#             wandb_run_name=f"ByT52021EN_to_{lang}_extra7ep",
#             logging_steps=LOGGING_STEPS,
#         )
#         print(f"[{task_idx}/{len(EXTRA_LANGS_TO_RUN)}] done: ByT52021EN_to_{lang} +7ep")


## 예측 및 official metric 평가

학습된 모델로 validation split을 예측하고 `summary.csv`에 Accuracy, ERR, TP/FP/FN 등을 저장합니다. 여기서도 subprocess 없이 `predict_seq2seq()`와 `evaluate_prediction_csv()`를 직접 호출합니다.


In [9]:
for task_idx, lang in enumerate(LANGS_TO_RUN, start=1):
    print("=" * 80)
    print(f"[{task_idx}/{len(LANGS_TO_RUN)}] predict/evaluate ByT52021EN_to_{lang}")
    model_label = f"ByT52021EN_to_{lang}"
    model_dir = f"outputs/byt5/{model_label}"
    pred_csv = f"outputs/byt5/{model_label}/predictions.csv"
    summary_csv = f"outputs/byt5/{model_label}/summary.csv"

    predict_seq2seq(
        model_name_or_path=model_dir,
        data_dir=f"{OUT_ROOT}/{lang}",
        output_csv=pred_csv,
        batch_size=PRED_BATCH_SIZE,
    )
    summary_df = evaluate_prediction_csv(
        pred_csv=pred_csv,
        out_csv=summary_csv,
        model=model_label,
        lang=lang,
    )
    display(summary_df)
    sync_to_drive(pred_csv)
    sync_to_drive(summary_csv)


[1/1] predict/evaluate ByT52021EN_to_vi


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


,model,lang,n,accuracy,ERR,TP,FP,FN,over_normalization,under_normalization,wrong_candidate,correct_keep,changed_accuracy,unchanged_accuracy
0,ByT52021EN_to_vi,vi,12924,0.90653,0.406388,1284,457,751,457,385,366,10432,0.630958,0.958031


[drive sync] outputs/byt5/ByT52021EN_to_vi/predictions.csv -> /drive/MyDrive/AI개론_박진영/lexnorm_outputs/byt5/ByT52021EN_to_vi/predictions.csv
[drive sync] outputs/byt5/ByT52021EN_to_vi/summary.csv -> /drive/MyDrive/AI개론_박진영/lexnorm_outputs/byt5/ByT52021EN_to_vi/summary.csv
